# Notebook 04 — Needleman-Wunsch: Global Sequence Alignment

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 04 of 18  
**Time estimate:** 90 minutes

> **Pass-3 target:** After completing this notebook, close it and implement
> `needleman_wunsch()` from scratch in a new file, without reference.
> This function must be reproducible from memory in any interview.

---
## Step 1 — Motivation

Needleman and Wunsch (1970) solved the global alignment problem: given two sequences
of any length, find the alignment of every character in both sequences that maximises
a score. This was the first bioinformatics dynamic programming algorithm and remains
the conceptual foundation of every modern aligner.

"Global" means the entire length of both sequences is aligned — appropriate when the
two sequences are of similar length and expected to be homologous end-to-end.

---
## Step 2 — Intuition

Like edit distance, but in reverse: instead of **minimizing** operations, we
**maximize** a score. Matches score positively (+1 typically), mismatches score
negatively (-1), and gaps score negatively (-2 or similar).

The key departure from edit distance: the gap penalty is tunable, and mismatches
can score differently from gaps. This allows the algorithm to express biological
priors ("a gap is worse than a mismatch").

The DP matrix is filled in the same way, but using a scoring system instead of cost
minimization. Traceback goes from the bottom-right corner back to the top-left.

---
## Step 3 — Biological Background

When two protein sequences are homologous (share an evolutionary ancestor), they tend
to be similar throughout their length. Gaps in an alignment represent insertion/deletion
events. Mismatches represent substitution events.

The scoring matrix encodes the relative likelihood of each substitution. For DNA,
simple +1/-1 works. For proteins, BLOSUM62 (NB06) gives a biologically calibrated
matrix. For now, we use simple integer scores to focus on the algorithm.

Global alignment is used for:
- Comparing entire genes
- Comparing entire protein sequences of similar length
- Benchmarking similarity before BLAST (which does local alignment)

---
## Step 4 — Mathematical Explanation

**Notation:**
- $s_1[1..n]$, $s_2[1..m]$ — the two sequences
- $F[i][j]$ — the optimal alignment score of the first $i$ chars of $s_1$ and first $j$ chars of $s_2$
- $\text{match}$ — score when $s_1[i] = s_2[j]$
- $\text{mismatch}$ — score when $s_1[i] \neq s_2[j]$
- $d$ — gap penalty (negative)

**Initialization:**
$$F[0][j] = j \cdot d, \quad F[i][0] = i \cdot d$$

**Recurrence:**
$$F[i][j] = \max \begin{cases}
F[i-1][j-1] + s(s_1[i], s_2[j]) & (\text{match or mismatch}) \\
F[i-1][j] + d & (\text{gap in } s_2 \text{ — delete from } s_1) \\
F[i][j-1] + d & (\text{gap in } s_1 \text{ — delete from } s_2)
\end{cases}$$

where $s(a, b) = \text{match if } a=b \text{ else mismatch}$.

**Answer:** $F[n][m]$ — the optimal global alignment score.

**Traceback:** from $F[n][m]$, move to the cell that generated the maximum at each step.
- Diagonal → aligned pair (match or mismatch)
- Up (decrease $i$) → gap in $s_2$
- Left (decrease $j$) → gap in $s_1$

---
## Step 5 — Computational Explanation

```
# Initialize boundary conditions
F[0][0] = 0
F[i][0] = i * gap   for i in 0..n
F[0][j] = j * gap   for j in 0..m

# Fill DP matrix (left-to-right, top-to-bottom)
For i in 1..n:
    For j in 1..m:
        score_sub = match if s1[i-1]==s2[j-1] else mismatch
        F[i][j] = max(
            F[i-1][j-1] + score_sub,  # diagonal
            F[i-1][j]   + gap,         # up
            F[i][j-1]   + gap          # left
        )

# Traceback from F[n][m]
i, j = n, m
align1, align2 = [], []
While i > 0 or j > 0:
    if i > 0 and j > 0 and F[i][j] came from diagonal:
        align1 += s1[i-1]; align2 += s2[j-1]; i--; j--
    elif i > 0 and F[i][j] came from up:
        align1 += s1[i-1]; align2 += '-'; i--
    else:
        align1 += '-'; align2 += s2[j-1]; j--

Return F[n][m], reversed(align1), reversed(align2)
```

In [ ]:
# Step 6 — Python Implementation
import numpy as np
from typing import Tuple


def needleman_wunsch(
    seq1: str,
    seq2: str,
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2
) -> Tuple[int, str, str]:
    n, m = len(seq1), len(seq2)
    F = np.zeros((n + 1, m + 1), dtype=int)

    # Initialize boundary
    for i in range(n + 1):
        F[i, 0] = i * gap
    for j in range(m + 1):
        F[0, j] = j * gap

    # Fill
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            score_sub = match if seq1[i - 1] == seq2[j - 1] else mismatch
            F[i, j] = max(
                F[i - 1, j - 1] + score_sub,  # diagonal
                F[i - 1, j] + gap,              # up (gap in seq2)
                F[i, j - 1] + gap               # left (gap in seq1)
            )

    # Traceback
    align1, align2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            score_sub = match if seq1[i - 1] == seq2[j - 1] else mismatch
            if F[i, j] == F[i - 1, j - 1] + score_sub:
                align1.append(seq1[i - 1])
                align2.append(seq2[j - 1])
                i -= 1; j -= 1
                continue
        if i > 0 and F[i, j] == F[i - 1, j] + gap:
            align1.append(seq1[i - 1])
            align2.append('-')
            i -= 1
        else:
            align1.append('-')
            align2.append(seq2[j - 1])
            j -= 1

    a1 = ''.join(reversed(align1))
    a2 = ''.join(reversed(align2))
    return int(F[n, m]), a1, a2


def print_alignment(seq1_aligned: str, seq2_aligned: str, match_char='|', mismatch_char='.'):
    middle = ''
    for a, b in zip(seq1_aligned, seq2_aligned):
        if a == b:
            middle += match_char
        elif a == '-' or b == '-':
            middle += ' '
        else:
            middle += mismatch_char
    print(f"seq1: {seq1_aligned}")
    print(f"      {middle}")
    print(f"seq2: {seq2_aligned}")


# Tests
print("=== Test 1: Simple ===\n")
score, a1, a2 = needleman_wunsch('ATCG', 'ATCG')
print(f"Score: {score} (expect 4)")
print_alignment(a1, a2)

print("\n=== Test 2: One mismatch ===\n")
score, a1, a2 = needleman_wunsch('ATCG', 'TTCG')
print(f"Score: {score} (expect 2: 3 matches + 1 mismatch = 3*1 + 1*(-1) = 2)")
print_alignment(a1, a2)

print("\n=== Test 3: Gap needed ===\n")
score, a1, a2 = needleman_wunsch('ATCG', 'ATCGTT')
print(f"Score: {score}")
print_alignment(a1, a2)

print("\n=== Test 4: Classic NW example ===\n")
score, a1, a2 = needleman_wunsch('GCATGCU', 'GATTACA')
print(f"Score: {score}")
print_alignment(a1, a2)

In [ ]:
# Validate against Biopython
from Bio import Align

aligner = Align.PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -2

test_pairs = [
    ('ATCG', 'ATCG'),
    ('ATCG', 'TTCG'),
    ('GCATGCU', 'GATTACA'),
    ('AGTACGCA', 'TATGC'),
]

print("Validation against Biopython PairwiseAligner:")
print(f"{'Seq1':>15} {'Seq2':>15} {'Our score':>10} {'Bio score':>10} {'Match':>7}")
all_match = True
for s1, s2 in test_pairs:
    our_score, _, _ = needleman_wunsch(s1, s2)
    bio_score = aligner.score(s1, s2)
    ok = abs(our_score - bio_score) < 0.1
    if not ok:
        all_match = False
    print(f"{s1:>15} {s2:>15} {our_score:>10} {bio_score:>10.1f} {'OK' if ok else 'FAIL':>7}")

print(f"\nAll match: {all_match}")

In [ ]:
# Step 7 — Visualization: NW DP matrix with traceback
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def visualize_nw_matrix(seq1, seq2, match=1, mismatch=-1, gap=-2):
    n, m = len(seq1), len(seq2)
    F = np.zeros((n + 1, m + 1), dtype=int)

    for i in range(n + 1):
        F[i, 0] = i * gap
    for j in range(m + 1):
        F[0, j] = j * gap

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sc = match if seq1[i-1] == seq2[j-1] else mismatch
            F[i, j] = max(F[i-1,j-1]+sc, F[i-1,j]+gap, F[i,j-1]+gap)

    # Traceback
    path = []
    i, j = n, m
    align1, align2 = [], []
    while i > 0 or j > 0:
        path.append((i, j))
        if i > 0 and j > 0:
            sc = match if seq1[i-1] == seq2[j-1] else mismatch
            if F[i, j] == F[i-1, j-1] + sc:
                align1.append(seq1[i-1]); align2.append(seq2[j-1])
                i -= 1; j -= 1; continue
        if i > 0 and F[i, j] == F[i-1, j] + gap:
            align1.append(seq1[i-1]); align2.append('-'); i -= 1
        else:
            align1.append('-'); align2.append(seq2[j-1]); j -= 1
    path.append((0, 0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    im = ax.imshow(F, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(m + 1))
    ax.set_xticklabels(['-'] + list(seq2), fontsize=12, fontweight='bold')
    ax.set_yticks(range(n + 1))
    ax.set_yticklabels(['-'] + list(seq1), fontsize=12, fontweight='bold')
    for ii in range(n + 1):
        for jj in range(m + 1):
            ax.text(jj, ii, F[ii, jj], ha='center', va='center', fontsize=9)

    px = [c[1] for c in path]
    py = [c[0] for c in path]
    ax.plot(px, py, 'ko-', markersize=8, lw=2, label='Traceback')
    ax.set_title(f'Needleman-Wunsch DP Matrix\n"{seq1}" vs "{seq2}" | Score={F[n,m]}')
    plt.colorbar(im, ax=ax)

    # Alignment display
    a1 = ''.join(reversed(align1))
    a2 = ''.join(reversed(align2))
    ax2 = axes[1]
    ax2.axis('off')
    mid = ''
    for ca, cb in zip(a1, a2):
        if ca == cb: mid += '|'
        elif ca == '-' or cb == '-': mid += ' '
        else: mid += '.'

    text = (
        f"Score: {F[n, m]}\n"
        f"Parameters: match={match}, mismatch={mismatch}, gap={gap}\n\n"
        f"Alignment:\n"
        f"{a1}\n"
        f"{mid}\n"
        f"{a2}\n\n"
        f"Matches: {mid.count('|')}\n"
        f"Mismatches: {mid.count('.')}\n"
        f"Gaps: {mid.count(' ')}"
    )
    ax2.text(0.05, 0.95, text, transform=ax2.transAxes, fontsize=12,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax2.set_title('Alignment result')

    plt.tight_layout()
    plt.savefig('needleman_wunsch.png', dpi=150, bbox_inches='tight')
    plt.show()


visualize_nw_matrix('GCATGCU', 'GATTACA')

In [ ]:
# Effect of gap penalty on alignment
seq1 = 'ATCGATCGATCG'
seq2 = 'ATCGTTTGATCG'

print("Effect of gap penalty on alignment of:")
print(f"  seq1: {seq1}")
print(f"  seq2: {seq2}")
print()

for gap_pen in [-1, -2, -4, -8]:
    score, a1, a2 = needleman_wunsch(seq1, seq2, gap=gap_pen)
    mid = ''.join('|' if a == b else (' ' if '-' in (a, b) else '.') for a, b in zip(a1, a2))
    print(f"gap={gap_pen:>3}: score={score:>4}")
    print(f"  {a1}")
    print(f"  {mid}")
    print(f"  {a2}")
    print()

---
## Step 8 — Exercises

See `exercises/04_nw_exercises.md`.

1. Implement NW from scratch (without looking at the code above) and verify
   against Biopython on 3 test pairs.
2. Align 'AGTACGCA' and 'TATGC' with match=2, mismatch=-1, gap=-2. What is the
   score and alignment? Draw the DP matrix by hand.
3. What happens if gap=0? Why is that biologically unreasonable?
4. Two sequences are identical. What is the NW score as a function of length $n$
   and match score $m$?
5. Measure the runtime of your implementation for sequences of length 50, 100, 200,
   400 bp. Plot and verify the O(n²) scaling.

---
## Step 10 — Quiz

1. What are the two differences between Levenshtein edit distance and NW alignment?
2. What does it mean for an alignment to be "global"?
3. Which direction does the traceback go?
4. If both Up and Diagonal give the same maximum value, which one do you pick? Does
   the choice matter for the score?
5. What is the space complexity of NW, and how can you reduce it?

---
## Step 12 — Reflection

> **Pass-3 rehearsal:** Close this notebook. Implement `needleman_wunsch(seq1, seq2,
> match, mismatch, gap) -> (score, aligned1, aligned2)` from scratch in a new cell.
> Open the notebook only to check your result, not to guide your implementation.

> *[Write one paragraph: what is the intuition behind the NW recurrence? Why does
> computing from top-left to bottom-right guarantee that each cell's value is optimal?]*

---
## Step 13 — Papers Referenced

- Needleman & Wunsch (1970) "A general method applicable to the search for
  similarities in the amino acid sequence of two proteins." *J Mol Biol* 48(3).
  The original paper. Pass-3 target — re-derive the recurrence after reading.

---
*Next: `05_smith_waterman_local_alignment.ipynb`*